# Status — read this first

**Scope changed from the original plan.** This started as a per-*step* faithfulness
metric for CoIG's editing chains (ARM). Two open-weight instruction editors
(InstructPix2Pix, MagicBrush) were tested and both failed to add a localized item to a
person — global color washes or near-no-ops instead. Two independent checkpoints
failing the same way was treated as a real result, not bad luck, so the study was
**reframed to single-shot compositional attribute binding**: does SD1.5's cross-attention
correctly bind each attribute to the right subject when generating a whole CoIG-style
prompt (multiple subjects, multiple attributes) in one shot — no editing chain.

This is closer to what Attend-and-Excite's own paper actually validated its method on.
It answers a different, narrower question than the original per-step plan: attribute
binding fidelity in one-shot generation, not chain-level faithfulness under CoIG's
compositional lock.

**Validated so far** (§4): on `"a photo of a barista wearing a red apron and a cyclist
wearing a yellow helmet"` — "red apron" bound correctly (`auc_margin +0.47`), but
"yellow helmet" bound to the *wrong* subject: its attention correlated more with the
apron's location (`auc_null 0.80`) than with the actual helmet (`auc 0.59`), a
**negative margin (-0.21)**. That's a real, structured incorrect-attribute-binding case
— exactly the failure mode Attend-and-Excite names — caught on the first real test.

**What's not done:** only one compositional prompt has been scored. No real CoIG items
are in `PROMPT_SPECS` yet (§6) — that's the next work. Read §6's token-budget note
before writing prompts; it's the part most likely to trip up a fresh run.

**Dead ends, so you don't re-walk them:** IP2P and MagicBrush for editing chains (see
above). Don't re-attempt without a stronger backend (FLUX.1 Kontext is the next
candidate, but it's a Diffusion Transformer — the attention-hooking code in §1 targets
SD1.5's UNet and does not transfer without real porting work, and it likely won't fit a
free T4).

# Spatial-Semantic Alignment (SSA) for CoIG-style prompts

Measures whether the model's **cross-attention during generation** actually targets the
object each attribute phrase names — an *internal* faithfulness signal, as opposed to
Causal Relevance, which only inspects the final image and is confounded by the
compositional lock (pilot result: Real 0.83 = Shuffled 0.83 persistence, so it can't
distinguish a true attribute-introduction step from a false one).

Method: Attend-and-Excite's inspection mechanism ([arXiv:2301.13826](https://arxiv.org/abs/2301.13826)),
used to **measure** rather than to **correct**.

## Design decision baked in

**Null condition.** Every score is paired with a **mismatched-mask** control: the same
attention map, scored against a *different* subject's mask in the same image. A raw
correlation has no meaning without a floor — this mirrors the pilot's Substituted
condition, which is what made that result defensible.

## Backend

CoIG's ARM runs on hosted Gemini, which exposes no attention maps, so this uses
open-weight **Stable Diffusion 1.5** — the exact architecture Attend-and-Excite's
attention code targets, so the hooking transfers directly with no porting.

> **Results describe SD1.5's attribute binding, not Gemini's or CoIG's ARM.** This is a
> deliberate scope narrowing (see Status cell above), not a stand-in for the ARM pipeline.

**Runtime:** ~2 GB VRAM in fp16 (SD1.5 + CLIPSeg only — no editor loaded). A free Colab
T4 is enough — set *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!pip -q install "diffusers>=0.31" "transformers>=4.44" accelerate safetensors scikit-learn

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Runtime > Change runtime type > T4 GPU.\n"
    "CPU-only diffusion runs minutes per image and is not usable for this study."
)
props = torch.cuda.get_device_properties(0)
print(f"{props.name}  |  {props.total_memory / 1e9:.1f} GB")

In [ ]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from dataclasses import dataclass
from PIL import Image

DEVICE = "cuda"
DTYPE = torch.float16
ATTN_RES = 16   # Attend-and-Excite uses the 16x16 cross-attention maps
SEED = 42

## 1 · Attention capture

diffusers' default `AttnProcessor2_0` calls fused scaled-dot-product attention, which
never materializes the attention matrix. We swap in a processor that computes attention
explicitly so the probabilities can be recorded.

Only **cross-attention** (`attn2`) processors are replaced. Self-attention keeps fused
SDPA — at 64×64 latent resolution an explicit self-attention matrix would be 4096×4096
per head and would OOM a T4.

For SD1.5 text→image, classifier-free guidance batches as `[uncond, cond]` — the
conditional branch is index **1 of 2**. (This mattered more when an editor with a
different batch layout was also in play; with only txt2img now, it's fixed and doesn't
need re-deriving per pipeline.)

In [ ]:
from diffusers.models.attention_processor import Attention


class AttentionStore:
    # Accumulates a running mean of cross-attention maps over layers x timesteps.
    def __init__(self, num_chunks, cond_index, attn_res=ATTN_RES):
        self.num_chunks = num_chunks
        self.cond_index = cond_index
        self.n_positions = attn_res * attn_res
        self.reset()

    def reset(self):
        self._sum = None
        self._count = 0

    def record(self, attention_probs, heads):
        # attention_probs: (num_chunks * heads, hw, 77)
        if attention_probs.shape[1] != self.n_positions:
            return                                  # wrong resolution, skip
        start = self.cond_index * heads
        maps = attention_probs[start:start + heads]  # conditional branch only
        maps = maps.mean(0).float()                  # (hw, 77), mean over heads
        self._sum = maps if self._sum is None else self._sum + maps
        self._count += 1

    def averaged(self):
        assert self._count, (
            "No attention captured. Check that ATTN_RES matches a real UNet "
            "resolution and that install_capture() ran on this pipeline's unet."
        )
        return (self._sum / self._count).cpu().numpy()


class CaptureProcessor:
    def __init__(self, store):
        self.store = store

    def __call__(self, attn: Attention, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, **kwargs):
        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim
        if input_ndim == 4:
            b, c, h, w = hidden_states.shape
            hidden_states = hidden_states.view(b, c, h * w).transpose(1, 2)

        is_cross = encoder_hidden_states is not None
        ctx = hidden_states if encoder_hidden_states is None else encoder_hidden_states
        batch_size, seq_len, _ = ctx.shape
        attention_mask = attn.prepare_attention_mask(attention_mask, seq_len, batch_size)

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)
        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)
        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        query = attn.head_to_batch_dim(query)
        key = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)

        attention_probs = attn.get_attention_scores(query, key, attention_mask)
        if is_cross:
            self.store.record(attention_probs, attn.heads)

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(b, c, h, w)
        if attn.residual_connection:
            hidden_states = hidden_states + residual
        return hidden_states / attn.rescale_output_factor


def install_capture(unet, store):
    # Replace only cross-attention processors; leave self-attention on fused SDPA.
    procs = {}
    for name, existing in unet.attn_processors.items():
        procs[name] = CaptureProcessor(store) if name.endswith("attn2.processor") else existing
    unet.set_attn_processor(procs)

In [ ]:
from diffusers import StableDiffusionPipeline

# runwayml/stable-diffusion-v1-5 was removed in 2024; these are the live mirrors.
SD15_CANDIDATES = [
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "sd-legacy/stable-diffusion-v1-5",
]

txt2img = None
for repo in SD15_CANDIDATES:
    try:
        txt2img = StableDiffusionPipeline.from_pretrained(
            repo, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False
        ).to(DEVICE)
        print("loaded txt2img:", repo)
        break
    except Exception as e:
        print(f"  {repo} unavailable ({type(e).__name__})")
assert txt2img is not None, "No SD1.5 mirror loaded -- check HF availability."
txt2img.set_progress_bar_config(disable=True)

In [ ]:
def token_indices(tokenizer, prompt, phrase):
    # Positions of `phrase` within the padded 77-token CLIP sequence.
    ids = tokenizer(prompt, padding="max_length", max_length=77, truncation=True).input_ids
    target = tokenizer(phrase, add_special_tokens=False).input_ids
    if not target:
        raise ValueError(f"phrase {phrase!r} tokenized to nothing")
    for i in range(len(ids) - len(target) + 1):
        if ids[i:i + len(target)] == target:
            return list(range(i, i + len(target)))
    raise ValueError(
        f"phrase {phrase!r} not found in prompt tokens -- it must appear "
        f"verbatim in the prompt text"
    )


@dataclass
class StepResult:
    image: Image.Image
    attn: np.ndarray   # (hw, 77), mean over heads x layers x timesteps


def run_initial(prompt, steps=30, guidance=7.5):
    store = AttentionStore(num_chunks=2, cond_index=1)      # [uncond, cond]
    install_capture(txt2img.unet, store)
    g = torch.Generator(DEVICE).manual_seed(SEED)
    img = txt2img(prompt, num_inference_steps=steps,
                  guidance_scale=guidance, generator=g).images[0]
    return StepResult(img, store.averaged())


def phrase_attention(attn, tokenizer, prompt, phrase, res=ATTN_RES):
    idx = token_indices(tokenizer, prompt, phrase)
    m = attn[:, idx].mean(1).reshape(res, res)
    return (m - m.min()) / (m.max() - m.min() + 1e-8)

## 2 · Object masks (CLIPSeg)

Text-promptable segmentation, light enough to sit alongside SD1.5 on a T4.

> **Validate this before trusting any score (§5).** A bad mask is indistinguishable from
> bad attention inside a correlation — both just lower the number. Fine attributes
> ("mustache", "headphones") are exactly where CLIPSeg is weakest, and those are the
> attributes CoIG's EC prompts are full of.

In [ ]:
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

seg_proc = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
seg_model = CLIPSegForImageSegmentation.from_pretrained(
    "CIDAS/clipseg-rd64-refined"
).to(DEVICE).eval()


@torch.no_grad()
def object_mask(image, phrase, res=ATTN_RES):
    inp = seg_proc(text=[phrase], images=[image], padding=True,
                   return_tensors="pt").to(DEVICE)
    logits = seg_model(**inp).logits
    prob = torch.sigmoid(logits).float()
    while prob.ndim < 4:
        prob = prob[None]
    m = F.interpolate(prob, size=(res, res), mode="area")[0, 0]
    return m.cpu().numpy()

## 3 · The metric

Two statistics, because they fail differently:

- **`pearson`** — continuous attention vs. a near-binary mask is effectively
  point-biserial correlation. Interpretable, but sensitive to how attention is
  normalized per prompt.
- **`auc`** — asks only whether in-mask pixels *rank* above out-of-mask pixels.
  Scale-free, so it survives normalization differences across prompts and token counts.

**Report both.** They diverge when attention is correctly placed but diffuse.

The mask threshold is the **top 20% by value** (`np.quantile(..., 0.80)`), not a fixed
`0.5` cutoff — CLIPSeg's raw sigmoid output for a small object is often well under 0.5
everywhere even when it's correctly localized (the mask is *relatively* brightest on the
object, not *absolutely* confident), so a fixed threshold was producing all-zero binary
masks and `NaN` AUCs. The mask is downsampled to attention's native 16×16 rather than
upsampling attention to image resolution — upsampling invents spatial detail that
inflates apparent agreement.

In [ ]:
from sklearn.metrics import roc_auc_score


def ssa(attn_map, mask, quantile=0.80):
    a = attn_map.ravel()
    m = mask.ravel()
    binary = (m > np.quantile(m, quantile)).astype(int)   # top 20% = object
    pearson = float(np.corrcoef(a, m)[0, 1])
    auc = (float(roc_auc_score(binary, a))
           if 0 < binary.sum() < len(binary) else float("nan"))
    return pearson, auc


def score_compositional(result, prompt, tokenizer, pairs):
    '''pairs: list of (target_phrase, distractor_phrase) tuples. Each target/distractor
    must appear verbatim in `prompt`. distractor should name a DIFFERENT subject's
    attribute in the same image -- the null condition.'''
    rows = []
    for target, distractor in pairs:
        amap = phrase_attention(result.attn, tokenizer, prompt, target)
        p_t, a_t = ssa(amap, object_mask(result.image, target))
        p_n, a_n = ssa(amap, object_mask(result.image, distractor))
        rows.append(dict(
            target=target, distractor=distractor,
            pearson=p_t, auc=a_t,
            pearson_null=p_n, auc_null=a_n,
            auc_margin=a_t - a_n,
        ))
    return pd.DataFrame(rows)

## 4 · Feasibility run — one compositional prompt

Two subjects, two attributes, generated in **one shot** — this is the harder,
competitive case where binding failures actually show up (two separate single-object
generations would each trivially "win," since there's no competing subject to bind the
wrong attribute to).

Target and distractor phrases must appear **verbatim** in the prompt — `token_indices`
locates them in the tokenized text. Expect ~20–30s on a T4.

In [ ]:
prompt = "a photo of a barista wearing a red apron and a cyclist wearing a yellow helmet"
r = run_initial(prompt)

pairs = [("red apron", "yellow helmet"), ("yellow helmet", "red apron")]
score_compositional(r, prompt, txt2img.tokenizer, pairs)

In [ ]:
def show_step(result, prompt, phrase, tokenizer):
    amap = phrase_attention(result.attn, tokenizer, prompt, phrase)
    mask = object_mask(result.image, phrase)
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
    ax[0].imshow(result.image);         ax[0].set_title("output")
    ax[1].imshow(amap, cmap="inferno"); ax[1].set_title(f"attention: {phrase}")
    ax[2].imshow(mask, cmap="viridis"); ax[2].set_title(f"CLIPSeg mask: {phrase}")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()


for target, _ in pairs:
    show_step(r, prompt, target, txt2img.tokenizer)

## 5 · Validate the segmenter — before believing any number

Check the "output" panel first for **catastrophic neglect** (Attend-and-Excite's other
named failure mode) — did both subjects actually get generated? If one is missing,
every score for it is meaningless; there's no correct region for attention or the mask
to agree on, and both will spuriously latch onto the same wrong region rather than
failing loudly.

Then check the mask panel against the output. If the **mask** doesn't sit on the object,
the SSA score for that phrase measures segmentation error, not attention — swap the
phrase for something CLIPSeg handles better, or switch to Grounded-SAM. Do not skip this
and then interpret a low score as unfaithfulness.

## 6 · Run real CoIG prompts

Fill in `PROMPT_SPECS` below. Each entry is one CoIG item scored as a single
compositional generation (no chain, no editing — see the Status cell).

**The one piece needing your judgement, and the part most likely to trip up a fresh
run:** SD1.5's CLIP text encoder truncates at 77 tokens and handles short, direct
phrasing far better than paragraph prose. CoIG's actual `prompt` column text repeats
"Housekeeping Staff X" four-plus times and will burn the token budget fast, degrading
binding independent of anything the model would otherwise do. **Write a short paraphrase
by hand for each item** — don't feed in the verbatim CSV text. Example, from the
`pilot/pilot_prompts.csv` "Housekeeping Staff" item used throughout the earlier
debugging session:

- verbatim (CSV): *"Four Housekeeping Staff. Housekeeping Staff A has tote bag in
  hand. Housekeeping Staff B has mustache. Housekeeping Staff C has water bottle in
  hand. Housekeeping Staff D wears headphones. ..."*
- paraphrased (use this instead): `"four housekeeping staff: one with a tote bag, one
  with a mustache, one with a water bottle, one with headphones"`

For each item you need:
- `prompt` — the short paraphrase (must literally contain every `target`/`distractor` phrase)
- `pairs` — `(target, distractor)` tuples; distractor should be another subject's
  attribute in the *same* prompt, for the null condition. Include both directions
  (A vs B *and* B vs A) if you want symmetric coverage, as in §4.

In [ ]:
# from google.colab import files
# uploaded = files.upload()   # pilot_prompts.csv, if you want the real prompt text for reference

PROMPT_SPECS = [
    dict(
        item_index=13,
        prompt="four housekeeping staff: one with a tote bag, one with a mustache, "
               "one with a water bottle, one with headphones",
        pairs=[
            ("tote bag", "mustache"), ("mustache", "tote bag"),
            ("water bottle", "headphones"), ("headphones", "water bottle"),
        ],
    ),
    # add more items here, following the paraphrase note above
]


def run_prompt_spec(spec):
    result = run_initial(spec["prompt"])
    df = score_compositional(result, spec["prompt"], txt2img.tokenizer, spec["pairs"])
    df["item_index"] = spec["item_index"]
    return df, result


all_rows = []
for spec in PROMPT_SPECS:
    df, _ = run_prompt_spec(spec)
    all_rows.append(df)
    print("done item", spec["item_index"])

if all_rows:
    result_df = pd.concat(all_rows, ignore_index=True)
    result_df.to_csv("ssa_results.csv", index=False)
    print(result_df[["item_index", "target", "auc", "auc_null", "auc_margin"]])
    print("\nsaved -> ssa_results.csv")
else:
    print("PROMPT_SPECS is empty -- fill it in above.")

## Reading the result

The number that matters is **`auc_margin` = `auc` − `auc_null`** — how much better
attention hits the right subject's object than a different subject's object, in the same
image.

| pattern | reading |
|---|---|
| margin ≈ 0 | attention doesn't discriminate the right object from a wrong one — no binding signal detected (**check §5 first**; catastrophic neglect or a failing segmenter both produce this) |
| margin clearly > 0, `auc` > 0.5 | attention correctly binds the attribute to its subject |
| margin clearly **negative** | attention binds the attribute to the *wrong* subject — a real incorrect-binding failure (this happened for "yellow helmet" in §4: margin -0.21) |
| `auc` ≈ 0.5 and margin ≈ 0 | no signal at all — suspect a wiring bug before concluding anything about the model |

**The bug that produces convincing-looking noise:** scoring the unconditional CFG pass
instead of the conditional one. If results look like noise, try flipping `cond_index`
to `0` in `run_initial`'s `AttentionStore` and re-run one prompt. If the numbers change
character completely, that was it.

Because `auc_null` uses a real competing subject from the same image rather than random
pixels, it's a **conservative** floor — a positive margin against it is meaningful, and
so is a negative one.